In [1]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"
!wget $PREFIX/rag_helper.py
!wget $PREFIX/starter.py

--2026-07-17 17:35:23--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 240.0.163.62
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|240.0.163.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1814 (1.8K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   1.77K  --.-KB/s    in 0s      

2026-07-17 17:35:24 (3.98 MB/s) - ‘rag_helper.py.1’ saved [1814/1814]

--2026-07-17 17:35:24--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 240.0.163.62
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|240.0.163.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 907 [text/plain]
Saving to: ‘starter.py.1’

starter.py.1      

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model inside a `while True` loop.

Each iteration:
1. sends the full `messages` history to the model,
2. checks the response for any `function_call`,
3. runs the tool and appends the tool output to `messages`,
4. repeats.

It stops when `has_function_calls` stays `False` for a turn, meaning the model returned a final message with no more tool calls.


In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

## Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3 [x]
* 5
* 7

In [1]:
from rag_helper import RAGBase, INSTRUCTIONS, PROMPT_TEMPLATE

class RAGTraced(RAGBase):
    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        super().__init__(index, llm_client, instructions, prompt_template, model)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response =  super().llm(prompt)
            span.set_attribute("input_tokens", response.usage.input_tokens)
            span.set_attribute("output_tokens", response.usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

In [ ]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)


{
    "name": "search",
    "context": {
        "trace_id": "0xb01e1bcbdd616b7dc5ed8cfd92a33ce8",
        "span_id": "0x0e2f418cd7abe77a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x41dcb7db0c32d23f",
    "start_time": "2026-07-17T15:49:40.335766Z",
    "end_time": "2026-07-17T15:49:40.337130Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xb01e1bcbdd616b7dc5ed8cfd92a33ce8",
        "span_id": "0x13db402da592d559",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000 [x]
* 70000
* 700000

> These numbers vary between runs. Pick the closest option.

In [22]:
rag = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xcc7c29411fce62a8f02dd5a66b94c9cf",
        "span_id": "0x80386d612ded3c73",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe8fb0aecc13589e0",
    "start_time": "2026-07-17T16:07:12.966313Z",
    "end_time": "2026-07-17T16:07:12.969614Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xcc7c29411fce62a8f02dd5a66b94c9cf",
        "span_id": "0x58609fd01b19ea34",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms [x]
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often.

```json
{
    "name": "search",
    "context": {
        "trace_id": "0xb01e1bcbdd616b7dc5ed8cfd92a33ce8",
        "span_id": "0x0e2f418cd7abe77a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x41dcb7db0c32d23f",
    "start_time": "2026-07-17T15:49:40.335766Z",
    "end_time": "2026-07-17T15:49:40.337130Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xb01e1bcbdd616b7dc5ed8cfd92a33ce8",
        "span_id": "0x13db402da592d559",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x41dcb7db0c32d23f",
    "start_time": "2026-07-17T15:49:40.337927Z",
    "end_time": "2026-07-17T15:49:42.706512Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0xb01e1bcbdd616b7dc5ed8cfd92a33ce8",
        "span_id": "0x41dcb7db0c32d23f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-17T15:49:40.335729Z",
    "end_time": "2026-07-17T15:49:42.707479Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
```

In [23]:
search_span = {
    "name": "search",
    "context": {
        "trace_id": "0xcc7c29411fce62a8f02dd5a66b94c9cf",
        "span_id": "0x80386d612ded3c73",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe8fb0aecc13589e0",
    "start_time": "2026-07-17T16:07:12.966313Z",
    "end_time": "2026-07-17T16:07:12.969614Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

llm_span = {
    "name": "llm",
    "context": {
        "trace_id": "0xcc7c29411fce62a8f02dd5a66b94c9cf",
        "span_id": "0x58609fd01b19ea34",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xe8fb0aecc13589e0",
    "start_time": "2026-07-17T16:07:12.971277Z",
    "end_time": "2026-07-17T16:07:14.843196Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 91
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "0176b862-3f8c-4fb7-b68c-216117a1791f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

In [25]:
from datetime import datetime, timezone

start_time = datetime.fromisoformat(llm_span["start_time"].replace("Z", "+00:00"))
end_time = datetime.fromisoformat(llm_span["end_time"].replace("Z", "+00:00"))
(end_time - start_time).seconds


1

## Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't
save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than
Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there.
But we need to create a custom exporter. Instead of printing the spans,
it will save them to the database.

OTel calls the exporter through the same span processor we already use,
we just swap the destination.

Now we will create a custom exporter that saves each finished span to a
SQLite database. The exporter extends `SpanExporter`. It has the following methods:

- `export` method that receives a list of `ReadableSpan` objects
- `shutdown` and `force_flush` methods

Let's implement it:

```python
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
```

Replace the console exporter with this new exporter:

```python
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```

Re-run the query from Q1. Which span names appear in the `spans` table?

* Only `rag`
* `rag` and `llm`
* `rag`, `search`, and `llm` [x]
* `search`, `llm`, and `judge`

In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [6]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)


The loop keeps calling the model in a `while True` loop. After each model response, it checks whether the response included any `function_call` items:

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and continues the loop.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.


In [ ]:
conn = sqlite3.connect("traces.db")
rows = conn.execute("select * from spans;")
conn.commit()


In [ ]:
for row in rows:
    print(row)

('search', 1784304954200598000, 1784304954202613000, None, None, None)
('llm', 1784304954203625000, 1784304956307805000, 7111, 125, None)
('rag', 1784304954200548000, 1784304956308931000, None, None, None)


## Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

* `search`
* `llm` [x]
* They're all about the same

In [ ]:
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("select * from spans", conn)

In [6]:
df['diff'] = df['end_time'] - df['start_time']
df

,name,start_time,end_time,input_tokens,output_tokens,cost,diff
0,search,1784304954200598000,1784304954202613000,NaN,NaN,None,2015000
1,llm,1784304954203625000,1784304956307805000,7111.0,125.0,None,2104180000
2,rag,1784304954200548000,1784304956308931000,NaN,NaN,None,2108383000


## Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?

* They're identical [x]
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%

In [8]:
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("select * from spans", conn)

In [10]:
df[df['name'] == 'llm']

,name,start_time,end_time,input_tokens,output_tokens,cost
1,llm,1784304954203625000,1784304956307805000,7111.0,125.0,None
4,llm,1784305835152168000,1784305838354661000,7111.0,84.0,None
7,llm,1784305842092622000,1784305843786702000,7111.0,91.0,None
10,llm,1784305846599603000,1784305848134713000,7111.0,95.0,None
